# Export Lake GAN for DreamTales UI

This notebook exports the trained lake-background GAN into UI-friendly model files. The exported model is not a fixed image: the UI will sample fresh random noise from this same model for each scene.

Outputs:
- `DreamVision 2.0/models/lake_background_generator.pt`: generator-only PyTorch state dict.
- `DreamVision 2.0/models/lake_background_generator.torchscript.pt`: TorchScript model for simple UI inference.
- `DreamVision 2.0/outputs/lake_background_story_256/ui_export_test/*.png`: quick sanity-check samples.


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

project_root = Path.cwd().resolve().parent.parent
dreamvision_root = project_root / "DreamVision 2.0"
checkpoint_dir = dreamvision_root / "outputs" / "lake_background_story_256" / "checkpoints"
checkpoint_path = sorted(checkpoint_dir.glob("lake_story_epoch_*.pt"))[-1]
export_script = dreamvision_root / "scripts" / "export_lake_gan_for_ui.py"
model_dir = dreamvision_root / "models"
sample_dir = dreamvision_root / "outputs" / "lake_background_story_256" / "ui_export_test"

print("Project root:", project_root)
print("Latest checkpoint:", checkpoint_path)
print("Export script exists:", export_script.exists(), export_script)


## Export

Run the export script. This strips out the discriminator and optimizer state so the UI does not need the full training checkpoint.

In [ ]:
result = subprocess.run(
    [sys.executable, str(export_script)],
    cwd=project_root,
    text=True,
    capture_output=True,
    check=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)


## Verify Exported Files

In [ ]:
generator_state_path = model_dir / "lake_background_generator.pt"
torchscript_path = model_dir / "lake_background_generator.torchscript.pt"

for path in [generator_state_path, torchscript_path]:
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"{path.name}: {size_mb:.1f} MB")


## Preview Export Samples

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

sample_paths = sorted(sample_dir.glob("lake_ui_export_sample_*.png"))
fig, axes = plt.subplots(1, len(sample_paths), figsize=(4 * len(sample_paths), 4))
if len(sample_paths) == 1:
    axes = [axes]
for axis, sample_path in zip(axes, sample_paths):
    axis.imshow(Image.open(sample_path))
    axis.set_title(sample_path.name)
    axis.axis("off")
plt.show()


## UI Integration Notes

For the UI, use the TorchScript file when possible:

```python
generator = torch.jit.load("DreamVision 2.0/models/lake_background_generator.torchscript.pt", map_location="cpu")
noise = torch.randn(1, 256, 1, 1)
with torch.no_grad():
    image_tensor = generator(noise)
```

For each scene, create a new `noise = torch.randn(1, 256, 1, 1)` call. Same model, fresh noise, fresh lake image.

The raw generator output is normalized to `[-1, 1]`, so convert it back with `(image_tensor + 1) / 2` before saving/displaying.